In [1]:
import os
from huggingface_hub import login

os.environ["HF_TOKEN"] = input("Enter your HF token: ")
login(os.environ["HF_TOKEN"])

d:\My works\Medical LLM\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [5]:
from datasets import load_dataset
import json

ds = load_dataset("lavita/MedQuAD")
train = ds["train"]

In [6]:
with open("dataset_stage2_wikipedia_augmented.json", "r", encoding="utf-8") as f:
    diseases = json.load(f)

disease_names = {d["disease"].lower() for d in diseases}

In [7]:
qa_data = []

for item in train:
    question = (item.get("question") or "").strip()
    answer   = (item.get("answer") or "").strip()
    uri      = item.get("uri", "")
    source   = item.get("source", "medquad")

    qa_data.append({
        "question": question,
        "answer": answer,
        "source": source,
        "uri": uri
    })

In [8]:
with open("medquad_clean.json", "w", encoding="utf-8") as f:
    json.dump(qa_data, f, indent=4, ensure_ascii=False)

In [9]:
filtered_qa = []

for qa in qa_data:
    for disease in disease_names:
        if disease in qa["question"].lower():
            filtered_qa.append(qa)
            break

In [10]:
with open("dataset_stage3_qa_filtered.json", "w") as f:
    json.dump(filtered_qa, f, indent=4)